# 09 — Embeddings, attention scores, and VLM losses: a worked deep dive

This notebook opens up the math hidden inside the familiar word “attention.” We will follow a token's embedding through projections, scores, softmax, value mixing, residual paths, and gradients. Then we will derive and compare token cross-entropy, symmetric image-text contrastive loss, image-text matching loss, and a simple grounding loss.

The implementation uses small PyTorch tensors and explicit operations. It is a pedagogical single-head/multi-head example, not an exact reproduction of any named model. Shapes are written as `[batch, sequence, feature]` unless otherwise stated.

## 1. Token IDs are addresses; embeddings are learned rows

A tokenizer converts text to discrete IDs. The embedding table `E` has shape `[vocab_size, d_model]`; token ID `i` selects row `E[i]`. The row is not a hand-authored definition. It is a parameter adjusted by gradients from all downstream objectives. Shared subwords therefore receive learning signal from every context in which they occur. Padding IDs are special and commonly excluded from loss/attention.

The token vector is usually combined with position information. Addition is element-wise, so both must have the same width: `x_t = E[token_t] + P[t]`. In many models positions are injected through rotary/relative mechanisms inside attention rather than by this simple learned table. The learned table below is chosen for visibility.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

torch.manual_seed(23)
vocab = {'<pad>': 0, '<bos>': 1, 'red': 2, 'block': 3, 'left': 4, 'blue': 5, 'cup': 6}
id_to_token = {idx: token for token, idx in vocab.items()}
tokens = torch.tensor([[vocab['<bos>'], vocab['red'], vocab['block'], vocab['left']]])
embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=4, padding_idx=vocab['<pad>'])
position = nn.Parameter(torch.randn(1, 8, 4) * 0.1)
token_vectors = embedding(tokens)
x = token_vectors + position[:, :tokens.shape[1]]
print('ids:', tokens.tolist())
print('embedding table:', tuple(embedding.weight.shape))
print('token vectors:', tuple(token_vectors.shape), 'position-added x:', tuple(x.shape))
print('red embedding row:', embedding.weight[vocab['red']].detach().tolist())
assert torch.allclose(token_vectors[0, 1], embedding.weight[vocab['red']])

### Why context changes a token's representation

The initial lookup for `block` is the same row in both sentences. After attention, its contextual representation can differ because the query reads a different set of neighboring keys/values. Thus “embedding” can mean the static lookup vector, a position-added input vector, or a contextual hidden state; be precise about which one is being discussed.

An embedding table receives sparse lookup gradients: only rows used by this batch get direct lookup gradients (but weight decay or optimizer behavior can still affect other rows). Its embedding row also receives downstream gradient through attention, MLPs, and the output head. The following cell shows the lookup gradient sparsity.

In [ ]:
embedding.zero_grad(set_to_none=True)
embedding(tokens).sum().backward()
row_gradient_norms = embedding.weight.grad.norm(dim=1)
pd.DataFrame({'token': [id_to_token[i] for i in range(len(vocab))],
              'lookup_gradient_norm': row_gradient_norms.detach().numpy()})

## 2. Images become vectors in a compatible width

A vision encoder outputs visual features `V` with shape `[B, N, d_vision]`. A learned projection (sometimes an MLP, resampler, query transformer, or architecture-specific connector) maps these features into a language-model width `d_model`: `Z = V W_proj + b`. A shared width makes the vectors mathematically usable in the same transformer operations; it does **not** by itself make the image and text meanings aligned. Joint training supplies that signal.

For a decoder-only model, the projected image vectors may be inserted into the causal token sequence. For an encoder-decoder or cross-attention design, text queries can read visual keys and values without concatenating the streams. Image features may include position/coordinate information and multiple crops or resolutions.

In [ ]:
B, N, d_vision, d_model = 1, 3, 5, 4
visual_features = torch.randn(B, N, d_vision)
vision_to_lm = nn.Linear(d_vision, d_model)
visual_tokens = vision_to_lm(visual_features)
joint_sequence = torch.cat([visual_tokens, x], dim=1)
print('raw visual features:', tuple(visual_features.shape))
print('projected visual tokens:', tuple(visual_tokens.shape))
print('illustrative image-prefix + text sequence:', tuple(joint_sequence.shape))
assert visual_tokens.shape[-1] == x.shape[-1]

## 3. Query, key, value: three learned views of hidden states

For a hidden sequence `X ∈ R^(B×T×D)`, each attention head creates:

`Q = X W_Q`, `K = X W_K`, `V = X W_V`

A query asks what this position is looking for; keys describe what each source position offers for matching; values carry the content to mix. These are useful intuitions, not literal symbolic labels learned by the network. In cross-attention, Q can come from text while K and V come from images.

The raw compatibility score from query position `i` to key position `j` is the dot product `q_i · k_j`. For a single head, `QKᵀ` has shape `[B, T_query, T_key]`. Larger dot products produce larger softmax weights. A learned linear projection changes what directions count as compatible.

In [ ]:
torch.manual_seed(4)
D = 4
X = torch.tensor([[[1.0, 0.0, 1.0, 0.0],
                   [0.0, 1.0, 0.0, 1.0],
                   [1.0, 1.0, 0.0, 0.0]]])
Wq = nn.Parameter(torch.eye(D))
Wk = nn.Parameter(torch.eye(D))
Wv = nn.Parameter(torch.eye(D))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
raw_scores = Q @ K.transpose(-2, -1)
print('X, Q, K, V:', tuple(X.shape), tuple(Q.shape), tuple(K.shape), tuple(V.shape))
print('raw query-key dot products for this toy:', raw_scores[0])

## 4. Scale, mask, softmax: the attention distribution

Scaled dot-product attention computes:

`S = QKᵀ / √d_k`
`A = softmax(S + mask, dim=keys)`
`C = A V`

Dividing by `√d_k` controls score magnitude as key width grows: if query/key components have roughly unit variance, the unscaled dot product variance grows with `d_k`. Large-magnitude logits can make softmax extremely peaked, saturating its derivatives and making optimization harder. Scaling does not force attention to be uniform; the model can still learn selective scores.

For each query row, softmax runs across the **key positions** and the weights sum to 1. A causal mask sets future key logits to negative infinity before softmax; a padding mask blocks invalid key positions. Masking logits after softmax is wrong because it does not renormalize the remaining probabilities.

In [ ]:
d_k = Q.shape[-1]
unscaled = Q @ K.transpose(-2, -1)
scaled = unscaled / (d_k ** 0.5)
causal_blocked = torch.triu(torch.ones(X.shape[1], X.shape[1], dtype=torch.bool), diagonal=1)
masked_logits = scaled.masked_fill(causal_blocked, float('-inf'))
attention_weights = masked_logits.softmax(dim=-1)
context = attention_weights @ V
print('scaled scores before mask:\n', scaled[0])
print('causal attention weights:\n', attention_weights[0])
print('row sums:', attention_weights.sum(dim=-1).tolist())
print('context vectors:', context[0])
assert torch.allclose(attention_weights.sum(dim=-1), torch.ones(1, X.shape[1]))
assert attention_weights[0, 0, 1:].count_nonzero() == 0

### Numerical scale demonstration

The next cell holds the same random Q/K draws fixed and compares entropy before/after scaling. Entropy near `log(number_of_keys)` means diffuse weights; entropy near zero means a very peaked distribution. One random sample is an illustration, not a general empirical guarantee.

In [ ]:
torch.manual_seed(9)
q = torch.randn(1, 1, 64)
k = torch.randn(1, 12, 64)
scores_unscaled = q @ k.transpose(-2, -1)
scores_scaled = scores_unscaled / (64 ** 0.5)
p_unscaled = scores_unscaled.softmax(-1)
p_scaled = scores_scaled.softmax(-1)
def entropy(probabilities):
    return -(probabilities * probabilities.clamp_min(1e-12).log()).sum(dim=-1)
print('unscaled max weight / entropy:', float(p_unscaled.max()), float(entropy(p_unscaled)))
print('scaled max weight / entropy:  ', float(p_scaled.max()), float(entropy(p_scaled)))

## 5. Multi-head attention and the information path

With `h` heads, the model projects width `D` into head width `d_k = D/h` per head. Each head computes its own attention distribution and value mixture. The head results are concatenated back to width `D`, passed through an output projection `W_O`, then normally combined with a residual connection and normalization:

`MHA(X) = concat(C₁,…,Cₕ) W_O`
`Y = LayerNorm(X + MHA(X))`

Residuals preserve a path for the original token representation while attention adds contextual information. The MLP sublayer then transforms each position, with another residual path. In a pre-norm transformer, normalization placement differs, but the same idea of repeated attention/MLP updates applies. Gradients can flow from a loss at an output token backward through value mixing, softmax scores, Q/K projections, and the original embeddings.

In [ ]:
B, T, D, heads = 2, 5, 12, 3
head_dim = D // heads
hidden = torch.randn(B, T, D)
qkv = nn.Linear(D, 3 * D)
out_proj = nn.Linear(D, D)
q_raw, k_raw, v_raw = qkv(hidden).chunk(3, dim=-1)
def split_heads(tensor):
    return tensor.reshape(B, T, heads, head_dim).transpose(1, 2)
qh, kh, vh = map(split_heads, (q_raw, k_raw, v_raw))
weights = (qh @ kh.transpose(-2, -1) / (head_dim ** 0.5)).softmax(dim=-1)
per_head = weights @ vh
merged = per_head.transpose(1, 2).contiguous().reshape(B, T, D)
attended = out_proj(merged)
residual_output = nn.LayerNorm(D)(hidden + attended)
print('Q/K/V per head:', tuple(qh.shape), tuple(kh.shape), tuple(vh.shape))
print('weights [B, heads, query, key]:', tuple(weights.shape))
print('merged/output:', tuple(merged.shape), tuple(residual_output.shape))

## 6. Cross-modal attention: a text query reads visual memory

Suppose one text position encodes “where is the red block?” Its hidden state makes a query. Image patch features make keys and values. The text query-key scores give a distribution over image patches; the weighted visual values are added to/refined into the text stream. Multiple layers and heads can perform different useful reads, but attention maps alone are not reliable explanations of model reasoning.

The sequence lengths need not match: Q is `[B, T_text, d_k]`, K/V are `[B, N_image, d_k]`, scores are `[B, T_text, N_image]`, and the output is text-length `[B, T_text, d_v]`. This simple cell intentionally exposes the score and weight tensors.

In [ ]:
text_hidden = torch.randn(1, 2, 8, requires_grad=True)
image_hidden = torch.randn(1, 4, 6, requires_grad=True)
W_q = nn.Linear(8, 8, bias=False)
W_k = nn.Linear(6, 8, bias=False)
W_v = nn.Linear(6, 8, bias=False)
q = W_q(text_hidden)
k, v = W_k(image_hidden), W_v(image_hidden)
cross_scores = q @ k.transpose(-2, -1) / (8 ** 0.5)
cross_weights = cross_scores.softmax(dim=-1)
read_visual = cross_weights @ v
loss_probe = read_visual.square().mean()
loss_probe.backward()
print('cross scores:', tuple(cross_scores.shape), 'weights:', tuple(cross_weights.shape))
print('visual read:', tuple(read_visual.shape), 'image-feature gradient norm:', float(image_hidden.grad.norm()))
assert image_hidden.grad is not None and image_hidden.grad.norm() > 0

## 7. Categorical cross-entropy: a next-token loss from first principles

For logits `z ∈ R^V` and correct class `y`, softmax gives `p_j = exp(z_j)/Σ_k exp(z_k)`. Cross-entropy is `−log(p_y)`, equivalently `logsumexp(z) − z_y`. The derivative with respect to each logit is `p_j − 1[j=y]`: gradient descent raises the correct logit relative to alternatives. Use `logsumexp`/`log_softmax` rather than explicitly exponentiating large logits, to avoid numerical overflow.

For a batch of tokens, average over valid supervised tokens. In causal language modeling, logits at position `t` predict token `t+1`. An `ignore_index` (often `-100`) removes padding and optionally user/prompt tokens. Be careful about shifting exactly once: model architectures/loss APIs differ in whether the shift is internal.

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0]], requires_grad=True)
target = torch.tensor([0])
log_probabilities = logits.log_softmax(dim=-1)
manual_ce = -log_probabilities[0, target[0]]
library_ce = F.cross_entropy(logits, target)
manual_ce.backward()
print('probabilities:', logits.detach().softmax(-1).tolist())
print('manual CE:', float(manual_ce), 'cross_entropy:', float(library_ce))
print('dL/dlogits:', logits.grad.tolist())
assert torch.allclose(manual_ce.detach(), library_ce)

In [ ]:
# Four next-token predictions. -100 masks positions not included in the objective.
B, T, V = 2, 4, 7
token_logits = torch.randn(B, T, V, requires_grad=True)
token_ids = torch.tensor([[1, 2, 3, 0], [1, 4, 5, 6]])
input_ids, labels = token_ids[:, :-1], token_ids[:, 1:].clone()
labels[0, 2] = -100  # illustrative pad/ignored target
shifted_logits = token_logits[:, :-1, :]
per_position = F.cross_entropy(shifted_logits.reshape(-1, V), labels.reshape(-1),
                               ignore_index=-100, reduction='none').reshape_as(labels)
valid = labels.ne(-100)
mean_valid_loss = per_position[valid].mean()
print('model inputs:', tuple(input_ids.shape), 'prediction logits:', tuple(shifted_logits.shape))
print('labels:', labels.tolist(), 'valid targets:', int(valid.sum()), 'mean loss:', float(mean_valid_loss))

### Teacher forcing and inference are different paths

During training with teacher forcing, each position receives the ground-truth preceding tokens; the causal mask ensures it cannot inspect future tokens. At inference, preceding tokens are the model's own generated tokens. Exposure bias describes the train/inference distribution mismatch. Evaluation should use the actual generation procedure and stop rules, not only teacher-forced loss.

Perplexity is `exp(mean token NLL)` for a specified tokenization and dataset. It can compare language modeling under the same tokenizer/data protocol, but it does not measure grounding, answer correctness, calibration, or robot success.

## 8. Symmetric contrastive image-text loss (CLIP-style)

Encode each image and its paired text, project to a shared dimension, and L2-normalize: `u_i = f(image_i)/||f(image_i)||`, `v_i = g(text_i)/||g(text_i)||`. Similarity logits are `s_ij = u_i·v_j / τ`. With N pairs, the diagonal is positive; off-diagonal pairs are in-batch negatives. Cross-entropy classifies the correct text for every image, and a second direction classifies the correct image for every text. Their mean is the symmetric objective.

Temperature `τ` controls logit sharpness and is often learned (commonly via a constrained log-scale). A small τ magnifies similarities and gradients. Duplicated/semantically equivalent captions can become false negatives; distributed training changes the negative pool; some methods use multi-positive labels or debiased losses. Contrastive training learns retrieval-aligned representations, not necessarily fluent generation.

In [ ]:
torch.manual_seed(13)
N, d = 4, 6
image_z = F.normalize(torch.randn(N, d), dim=-1)
text_z = F.normalize(image_z + 0.25 * torch.randn(N, d), dim=-1)
logit_scale = nn.Parameter(torch.tensor(np.log(1 / 0.07), dtype=torch.float32))
similarity = logit_scale.exp() * (image_z @ text_z.T)
pair_labels = torch.arange(N)
image_to_text = F.cross_entropy(similarity, pair_labels)
text_to_image = F.cross_entropy(similarity.T, pair_labels)
contrastive_loss = (image_to_text + text_to_image) / 2
print('pairwise logits [image, text]:\n', similarity.detach())
print('directional losses:', float(image_to_text), float(text_to_image))
print('symmetric contrastive loss:', float(contrastive_loss))
print('retrieval R@1:', float((similarity.argmax(1) == pair_labels).float().mean()))
contrastive_loss.backward()
print('logit-scale gradient:', float(logit_scale.grad))

## 9. Other common objectives and what they supervise

- **Image-text matching (ITM):** a binary/multiclass classifier predicts whether a pair matches. Often trained with sampled hard negatives; it can learn fine-grained fusion but depends on negative quality.
- **Masked language modeling / masked image modeling:** predict hidden/corrupted portions. Common in encoder-style representation learning; not the same as next-token generation.
- **Caption/VQA/instruction SFT:** autoregressive cross-entropy on target answer tokens, usually masking prompt tokens. This teaches the response format and task behavior represented in curated data.
- **Grounding:** predict boxes/points/masks or text spans aligned to regions. Losses can combine L1 coordinate error with IoU/GIoU, classification, or segmentation terms. Coordinate normalization and conventions must be consistent.
- **Preference/alignment objectives:** compare chosen/rejected responses or optimize a reward signal. These depend on preference quality and can distort behavior if reward or sampling is flawed.

A training recipe may combine objectives with weights, schedules, or stages. Normalize/monitor components: one numerically large term can dominate gradients. Report both each raw loss and its weighted contribution; inspect gradient norms by module.

In [ ]:
# Image-text matching: two logits per pair (mismatch, match), one binary class label.
itm_logits = torch.tensor([[1.4, -0.2], [-0.3, 1.0], [0.1, 0.6]], requires_grad=True)
is_match = torch.tensor([0, 1, 1])
itm_loss = F.cross_entropy(itm_logits, is_match)

# Grounding example: normalized xyxy boxes; use smooth L1 for coordinate regression.
pred_boxes = torch.tensor([[0.20, 0.18, 0.62, 0.72], [0.1, 0.2, 0.4, 0.6]], requires_grad=True)
target_boxes = torch.tensor([[0.22, 0.20, 0.60, 0.70], [0.0, 0.2, 0.5, 0.6]])
box_loss = F.smooth_l1_loss(pred_boxes, target_boxes)

total = 0.7 * itm_loss + 2.0 * box_loss
total.backward()
print('ITM CE:', float(itm_loss), 'box smooth-L1:', float(box_loss), 'weighted total:', float(total))
print('box parameter gradient norm:', float(pred_boxes.grad.norm()))

## 10. Loss is a learning signal, not the final capability

Training minimizes a differentiable surrogate over sampled examples. Deployment asks whether behavior generalizes. A lower caption loss can coexist with hallucinations, poor OCR, prompt sensitivity, or image-independent answers. Evaluate using independent data, multiple task slices, image/text ablations, and per-example error inspection.

For robot data, action MSE can be deceptively high for valid multimodal action distributions or deceptively low while missing rare contact-critical events. Evaluate trajectories closed-loop in simulation before any hardware. Reward-based RL additionally faces exploration risk, sparse signals, reward hacking, and distribution shift; do not treat a scalar reward as a safety certificate.

## Exercises

1. Set `Wq`, `Wk`, `Wv` to different matrices and explain how the score matrix changes even when `X` is fixed.
2. Increase `d_k` while holding component variance roughly fixed. Compare raw and scaled score variance and softmax entropy over 20 seeds.
3. Remove the causal mask. Show that an early position can now attend to later positions and explain why next-token training would leak the answer.
4. In cross-attention, make one image key match one text query strongly. Confirm its probability rises and visualize the corresponding value contribution.
5. Compute token-level cross-entropy with one `-100` label. Compare averaging over all positions (incorrect) with averaging only valid targets.
6. Duplicate one caption in a contrastive batch. Explain why the duplicate may be treated as a false negative; implement a multi-positive target distribution.
7. Log raw and weighted ITM/box/language losses and gradient norms. What changes if you multiply one term by 100?
8. Compare teacher-forced next-token loss with greedy generation exact-match on held-out examples. Why are the metrics different?

## Summary: the full information/gradient path

`token ID → embedding row + position → hidden state → Q/K/V projections → scaled compatibility scores → masked softmax weights → value mixture → residual/MLP blocks → logits → cross-entropy`

Backpropagation traverses this path in reverse. In a multimodal model, the gradient can continue through the text-to-image attention read, visual projection, vision encoder, and ultimately image patch features. Whether those modules actually update depends on freezing, `requires_grad`, and the optimizer parameter groups.